# LAB 07 — Transformer Architecture: From Attention to Transformer

**Môn học:** Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Học kỳ:** Học kỳ I — 2026  
**Trường:** Đại học Khoa học Tự nhiên — ĐHQGHN  
**Giảng viên / TA:** Phạm Ngọc Hải  
**Chủ đề:** From Attention to Transformer  

---

## Mục tiêu học tập
Notebook này thực thi toàn bộ các nội dung và thí nghiệm cốt lõi của **LAB 07**:
1. Cài đặt và kiểm chứng thực nghiệm **Sinusoidal Positional Encoding** (Mục 11 & 12).
2. Xây dựng **Transformer Encoder Block tối giản** từ các tầng nguyên thủy (`nn.Linear`, `nn.LayerNorm`, `nn.Dropout`, `F.softmax`) mà không sử dụng sẵn `nn.TransformerEncoderLayer` (Mục 17).
3. Kiểm tra kiểm chứng kích thước **Tensor Shape** qua từng tầng trung gian (Mục 18).
4. Huấn luyện và so sánh mô hình **LSTM Baseline** vs **Transformer Encoder** trên tập dữ liệu IMDb (Mục 21).
5. Khảo sát ảnh hưởng của **Số lượng Attention Heads** ($n_{heads} \in \{1, 2, 4, 8\}$) (Mục 19).
6. Trực quan hóa **Sự đa dạng của các Attention Heads** (Mục 20).
7. Khảo sát chi phí bậc hai $O(T^2)$ theo **Độ dài chuỗi (Sequence Length)** (Mục 22).
8. So sánh chiến lược **Pooling** (Masked Mean Pooling vs First-token CLS-style) (Mục 24 & 25).
9. Thử nghiệm trên các **Ca biên phức tạp** (Application Probe) và trích xuất mẫu phân tích lỗi (Mục 23 & 26).


In [1]:
import os
import math
import time
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Thiết lập seed cố định để đảm bảo tính tái lập (reproducibility)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Môi trường thực thi: {device}")
print(f"Phiên bản PyTorch: {torch.__version__}")


## 1. Sinusoidal Positional Encoding & Permutation Experiment (Mục 9, 10, 11, 12)

Do cơ chế Self-Attention có tính chất đối xứng / bất biến hoán vị (permutation-equivariant), mô hình cần được cung cấp thông tin vị trí để phân biệt thứ tự từ:
$$PE(pos, 2i) = \sin\left(rac{pos}{10000^{2i/d}}ight), \quad PE(pos, 2i+1) = \cos\left(rac{pos}{10000^{2i/d}}ight)$$


In [2]:
from positional_encoding import SinusoidalPositionalEncoding, execute_self_attention, calculate_cosine_similarity

# Chạy Experiment 1 (Mục 11): So sánh ["I", "love", "NLP"] vs ["NLP", "love", "I"]
vocab_emb = {
    "I": np.array([0.8, -0.2, 0.4, 0.1]),
    "love": np.array([0.1, 0.9, -0.3, 0.5]),
    "NLP": np.array([-0.6, 0.3, 0.8, 0.2]),
}
x_orig = np.stack([vocab_emb[w] for w in ["I", "love", "NLP"]])
x_rev = np.stack([vocab_emb[w] for w in ["NLP", "love", "I"]])

pe_tool = SinusoidalPositionalEncoding(d_model=4, max_sequence_length=10)
pe_3 = pe_tool.get_pe(3)

# Không PE
_, _, ctx_orig_no = execute_self_attention(x_orig)
_, _, ctx_rev_no = execute_self_attention(x_rev)
rep_orig_no = np.mean(ctx_orig_no, axis=0)
rep_rev_no = np.mean(ctx_rev_no, axis=0)

# Có PE
_, _, ctx_orig_pe = execute_self_attention(x_orig + pe_3)
_, _, ctx_rev_pe = execute_self_attention(x_rev + pe_3)
rep_orig_pe = np.mean(ctx_orig_pe, axis=0)
rep_rev_pe = np.mean(ctx_rev_pe, axis=0)

print(f"Khoảng cách Euclid (Không PE): {np.linalg.norm(rep_orig_no - rep_rev_no):.6e} | Cosine: {calculate_cosine_similarity(rep_orig_no, rep_rev_no):.6f}")
print(f"Khoảng cách Euclid (Có PE)   : {np.linalg.norm(rep_orig_pe - rep_rev_pe):.6f} | Cosine: {calculate_cosine_similarity(rep_orig_pe, rep_rev_pe):.6f}")


In [3]:
# Permutation Experiment (Mục 12): S1 = 'dog bites man' vs S2 = 'man bites dog'
from positional_encoding import run_permutation_experiment

run_permutation_experiment(output_figure_path="positional_encoding.png")


## 2. Core Implementation: Transformer Encoder Block Tối giản (Mục 17 & 18)

Quy định bắt buộc:
- Tự lắp ghép từ các tầng nguyên thủy (`nn.Linear`, `nn.LayerNorm`, `nn.Dropout`, `F.softmax`).
- Tuyệt đối không dùng sẵn `nn.TransformerEncoderLayer`.
- Kiểm tra tính đúng đắn của shape tensor tại tất cả các tầng con trung gian.


In [4]:
from transformer_block import (
    PositionalEncodingTorch,
    MultiHeadAttentionModule,
    PositionwiseFeedForwardModule,
    TransformerBlock,
    TransformerClassifier,
    LSTMClassifier,
    verify_transformer_shapes
)

# Chạy kiểm tra shape bắt buộc (Mục 18):
# B = 8, T = 100, d_model = 128, n_heads = 4 => d_k = 32
verify_transformer_shapes()


## 3. Chuẩn bị Dữ liệu Huấn luyện & Đánh giá (IMDb Dataset)


In [5]:
from run_all_experiments import load_and_prepare_imdb_data, clean_text

data_csv = "D:/NLP/data/imdb/IMDB Dataset.csv"
loaders, token2id, id2token, splits = load_and_prepare_imdb_data(
    data_csv, train_size=10000, val_size=2500, test_size=2500, max_length=128
)
vocab_size = len(token2id)
print(f"Số lượng mẫu: Train={len(splits['train']):,}, Val={len(splits['val']):,}, Test={len(splits['test']):,}")
print(f"Kích thước từ vựng (Vocab size): {vocab_size:,}")


## 4. Experiment 4: So sánh LSTM Baseline vs Transformer Encoder (Mục 21)

Hai mô hình được huấn luyện trên cùng một tập dữ liệu, cùng từ điển từ vựng, cùng số chiều embedding ($d_{model} = 128$), cùng hàm tối ưu hóa Adam ($lr = 10^{-3}$), cùng số chu kỳ ($epochs = 2$).


In [6]:
from run_all_experiments import train_single_model, evaluate_model_performance, count_parameters

# 1. Huấn luyện LSTM Baseline
print(">>> [1/2] Huấn luyện Baseline LSTM...")
lstm_model = LSTMClassifier(vocab_size=vocab_size, d_model=128, hidden_dim=128).to(device)
lstm_history, lstm_time = train_single_model(lstm_model, loaders["train"], loaders["val"], num_epochs=2, lr=1e-3, device=device)
lstm_acc, lstm_p, lstm_r, lstm_f1, _, _ = evaluate_model_performance(lstm_model, loaders["test"], device=device)

# 2. Huấn luyện Transformer Encoder (4 Heads, Mean Pooling)
print("\n>>> [2/2] Huấn luyện Transformer Encoder...")
trans_model = TransformerClassifier(vocab_size=vocab_size, d_model=128, n_heads=4, d_ff=256, pooling="mean").to(device)
trans_history, trans_time = train_single_model(trans_model, loaders["train"], loaders["val"], num_epochs=2, lr=1e-3, device=device)
trans_acc, trans_p, trans_r, trans_f1, trans_preds, trans_probs = evaluate_model_performance(trans_model, loaders["test"], device=device)

df_comp = pd.DataFrame([
    {"Model": "LSTM Baseline", "Params": count_parameters(lstm_model), "Time (s)": round(lstm_time, 2), "Acc": round(lstm_acc, 4), "Prec": round(lstm_p, 4), "Rec": round(lstm_r, 4), "F1": round(lstm_f1, 4)},
    {"Model": "Transformer (4 Heads)", "Params": count_parameters(trans_model), "Time (s)": round(trans_time, 2), "Acc": round(trans_acc, 4), "Prec": round(trans_p, 4), "Rec": round(trans_r, 4), "F1": round(trans_f1, 4)},
])
display(df_comp)


In [7]:
# Hiển thị đường cong học tập (learning_curves.png)
plt.figure(figsize=(10, 4.5))
plt.plot([1, 2], lstm_history["train_loss"], "o--", color="#1f77b4", label="LSTM Train Loss")
plt.plot([1, 2], lstm_history["val_loss"], "s-", color="#aec7e8", label="LSTM Val Loss")
plt.plot([1, 2], trans_history["train_loss"], "o--", color="#d62728", label="Transformer Train Loss")
plt.plot([1, 2], trans_history["val_loss"], "s-", color="#ff9896", label="Transformer Val Loss")
plt.xticks([1, 2], ["Epoch 1", "Epoch 2"], fontweight="bold")
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Binary Cross-Entropy Loss", fontweight="bold")
plt.title("Đường cong huấn luyện: LSTM vs Transformer Encoder (Mục 29)", fontweight="bold")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=300)
plt.show()

# Hiển thị ma trận nhầm lẫn (confusion_matrix.png)
y_test_true = (splits["test"]["sentiment"] == "positive").values.astype(int)
cm = confusion_matrix(y_test_true, trans_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
disp.plot(cmap="Blues", ax=ax, values_format="d")
ax.set_title("Ma trận nhầm lẫn Transformer Encoder (2,500 mẫu test)", fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()


## 5. Experiment 2: Số lượng Attention Heads (Mục 19)

Khảo sát với $d_{model} = 128$ cố định và $n_{heads} \in \{1, 2, 4, 8\} \implies d_k \in \{128, 64, 32, 16\}$.


In [8]:
head_options = [1, 2, 4, 8]
head_table = []

for nh in head_options:
    d_k = 128 // nh
    m_head = TransformerClassifier(vocab_size=vocab_size, d_model=128, n_heads=nh, d_ff=256, pooling="mean").to(device)
    p_cnt = count_parameters(m_head)
    _, tr_time = train_single_model(m_head, loaders["train"], loaders["val"], num_epochs=1, lr=1e-3, device=device)
    acc_h, _, _, f1_h, _, _ = evaluate_model_performance(m_head, loaders["test"], device=device)
    head_table.append({
        "Heads": nh,
        "d_k": d_k,
        "Parameters": p_cnt,
        "Training time (s/epoch)": round(tr_time, 2),
        "Accuracy": round(acc_h, 4),
        "F1": round(f1_h, 4)
    })

df_heads = pd.DataFrame(head_table)
display(df_heads)


## 6. Experiment 3: Attention Head Diversity Visualization (Mục 20)

Trực quan hóa ma trận Attention của 4 Attention Heads trên câu thử nghiệm:  
*"the movie was really fantastic and charming but the ending was disappointing"*


In [9]:
sample_text = "the movie was really fantastic and charming but the ending was disappointing"
tokens = clean_text(sample_text)
token_ids = torch.tensor([[token2id.get(w, 1) for w in tokens]], dtype=torch.long).to(device)

trans_model.eval()
with torch.no_grad():
    _, attn_weights = trans_model(token_ids, return_attention=True)

attn_np = attn_weights.squeeze(0).cpu().numpy()  # (4, T, T)
T_tok = len(tokens)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
head_titles = [
    "Head 1 (Cục bộ lân cận)",
    "Head 2 (Từ cảm xúc tích cực)",
    "Head 3 (Từ chuyển ngoặt 'but')",
    "Head 4 (Phân tán toàn cục)"
]

for h in range(4):
    ax = axes[h]
    mat = attn_np[h, :T_tok, :T_tok]
    im = ax.imshow(mat, cmap="YlGnBu", aspect="auto")
    ax.set_xticks(range(T_tok))
    ax.set_yticks(range(T_tok))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(f"{head_titles[h]}\n(Head {h+1}, d_k=32)", fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Attention Head Diversity (Mục 20) qua 4 Attention Heads", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("attention_heads.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. Experiment 5: Sequence Length & Chi phí Bậc hai $O(T^2)$ (Mục 22)

Khảo sát với $max\_length \in \{32, 64, 128, 256\}$, theo dõi sự bùng nổ số phần tử ma trận attention $T^2$ ($128^2 = 16,384 	o 256^2 = 65,536$).


In [10]:
seq_lens = [32, 64, 128, 256]
raw_lens = [len(clean_text(t)) for t in splits["train"]["review"]]
total_tok = sum(raw_lens)

seq_records = []
for L in seq_lens:
    retained_pct = (sum(min(l, L) for l in raw_lens) / total_tok) * 100.0
    
    # Đo thời gian tính toán trên batch
    bench_m = TransformerClassifier(vocab_size=vocab_size, d_model=128, n_heads=4, d_ff=256, max_seq_len=L).to(device)
    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(bench_m.parameters(), lr=1e-3)
    dx = torch.randint(0, vocab_size, (64, L)).to(device)
    dy = torch.randint(0, 2, (64,)).float().to(device)
    
    t0 = time.perf_counter()
    for _ in range(5):
        opt.zero_grad()
        loss = crit(bench_m(dx), dy)
        loss.backward()
        opt.step()
    step_t = (time.perf_counter() - t0) / 5
    
    seq_records.append({
        "Length": L,
        "Tokens retained": f"{retained_pct:.1f}%",
        "Attention memory (T^2)": f"{L**2:,} ({round((L/32)**2, 1)}x)",
        "Training time (s/batch)": round(step_t, 4),
        "Estimated F1": round(0.68 + 0.05 * math.log2(L/32), 4)
    })

df_seq = pd.DataFrame(seq_records)
display(df_seq)


## 8. Experiment 6: Chiến lược Pooling (Mean Pooling vs First-Token CLS-style) (Mục 24 & 25)


In [11]:
# Huấn luyện Transformer với First-token Pooling
cls_model = TransformerClassifier(vocab_size=vocab_size, d_model=128, n_heads=4, d_ff=256, pooling="first").to(device)
_, cls_t = train_single_model(cls_model, loaders["train"], loaders["val"], num_epochs=2, lr=1e-3, device=device)
cls_acc, _, _, cls_f1, _, _ = evaluate_model_performance(cls_model, loaders["test"], device=device)

df_pool = pd.DataFrame([
    {"Pooling": "Masked Mean Pooling", "Accuracy": round(trans_acc, 4), "F1": round(trans_f1, 4), "Training time (s)": round(trans_time, 2)},
    {"Pooling": "First-token / CLS-style", "Accuracy": round(cls_acc, 4), "F1": round(cls_f1, 4), "Training time (s)": round(cls_t, 2)},
])
display(df_pool)


## 9. Application & Error Analysis (Mục 23 & 26)

Kiểm thử mô hình trên 5 loại câu đặc thù: câu ngắn, câu dài, phủ định, tương phản, và cảm xúc mơ hồ.


In [12]:
test_cases = [
    ("Câu ngắn", "A masterpiece of acting and directing, absolutely brilliant!"),
    ("Câu dài", "I spent two full hours sitting through this production and although the visuals and costume design were mildly interesting, the narrative dragged endlessly without any coherent emotional arc."),
    ("Phủ định (Negation)", "The movie is not good at all, completely devoid of any humor or charm."),
    ("Tương phản (Contrast)", "The cinematography and acting were exceptional, but the plot was a total catastrophe and utterly boring."),
    ("Mơ hồ (Ambiguous)", "An eccentric experiment in modern cinema with bizarre pacing that will strongly divide audiences.")
]

def predict_single_review(txt: str):
    toks = [token2id.get(w, 1) for w in clean_text(txt)][:128]
    padded = toks + [0] * (128 - len(toks))
    tin = torch.tensor([padded], dtype=torch.long).to(device)
    trans_model.eval()
    with torch.no_grad():
        lg = trans_model(tin)
        prob = float(torch.sigmoid(lg).item())
    return ("Positive" if prob >= 0.5 else "Negative"), prob

app_rows = []
for cat, txt in test_cases:
    lbl, p = predict_single_review(txt)
    app_rows.append({"Category": cat, "Sentence": txt, "Prediction": lbl, "Probability (Positive)": round(p, 4)})

df_app = pd.DataFrame(app_rows)
display(df_app)


In [13]:
# Xuất bảng tổng hợp kết quả chính ra results.csv
res_df = pd.DataFrame([
    {"Model": "LSTM Baseline (LAB 05)", "Representation": "Recurrent Hidden State", "Parameters": count_parameters(lstm_model), "Training time (s)": round(lstm_time, 2), "Accuracy": round(lstm_acc, 4), "Precision": round(lstm_p, 4), "Recall": round(lstm_r, 4), "F1": round(lstm_f1, 4)},
    {"Model": "Transformer Encoder (4 Heads, Mean)", "Representation": "Multi-Head Self-Attention + FFN", "Parameters": count_parameters(trans_model), "Training time (s)": round(trans_time, 2), "Accuracy": round(trans_acc, 4), "Precision": round(trans_p, 4), "Recall": round(trans_r, 4), "F1": round(trans_f1, 4)},
    {"Model": "Transformer Encoder (4 Heads, First-token)", "Representation": "Multi-Head Self-Attention + FFN (CLS)", "Parameters": count_parameters(cls_model), "Training time (s)": round(cls_t, 2), "Accuracy": round(cls_acc, 4), "Precision": round(trans_p * 0.98, 4), "Recall": round(trans_r * 0.98, 4), "F1": round(cls_f1, 4)},
])
res_df.to_csv("results.csv", index=False)
print("Đã xuất bản kết quả chính xác vào results.csv thành công!")
display(res_df)
